In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
def plot_one_loss(df, name, color):
    # Plot Train and Validation Loss
    plt.figure(figsize=(10, 5))
    plt.plot(df[name], label=name, color=color)
    #plt.plot(df['Val Loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    #plt.xlim(0,150)
    plt.title(name)
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
def plot_two_loss(df):    
    plt.figure(figsize=(10, 5))
    plt.plot(df['Train Loss'], label='Train Loss', color="orange")
    plt.plot(df['Val Loss'], label='Validation Loss', color="red")
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
def plot_metrics(df):
# Plot Validation Accuracy
    plt.figure(figsize=(10, 5))
    plt.plot(df['Val Accuracy'], label='Validation Accuracy', color='green')
    plt.plot(df['Val F1'], label='Validation F1', color='blue')
    plt.plot(df['Val AUC'], label='Validation AUC', color='purple')
    plt.plot((df['Val Accuracy']+df['Val F1']+df['Val AUC'])/3, label='Validation combined', color='pink')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.ylim(0,1)
    plt.title('Validation Accuracy')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
def plot_f1(df):    
    
    try:
        df_ = df[15:]
        max_f1_index = df_['Val F1'].idxmax()
        print("Best value saved for epoch: " + str(max_f1_index))
        print("Accuracy: " + str(df.loc[max_f1_index, 'Val Accuracy']))
        print("F1: " + str((df.loc[max_f1_index, 'Val F1'])))
    except:
        df_ = df
        max_f1_index = 0

    plt.figure(figsize=(10, 5))
    plt.plot(df['Train F1'], label='Training F1', color='yellow')
    plt.plot(df['Val F1'], label='Validation F1', color='blue')
    plt.xlabel('Epoch')
    
    if max_f1_index>0:
        plt.vlines(max_f1_index,0,1)
    
    plt.ylabel('F1 Macro')
    plt.ylim(0,1)
    plt.title('F1')

    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LinearRegression
from sklearn.metrics import brier_score_loss
from scipy.stats import ks_2samp, chi2


def format_p_value(p_value):
    if p_value < 0.001:
        return "<0.001"
    elif p_value < 0.05:
        return "<0.05"
    else:
        return f"{p_value:.2f}"
    
def hosmer_lemeshow_test(labels, predicted_probs, n_bins=10):
    data = pd.DataFrame({'label': labels, 'prob': predicted_probs})
    data['bin'] = pd.qcut(data['prob'], n_bins, duplicates='drop')
    
    observed = data.groupby('bin')['label'].sum()
    expected = data.groupby('bin')['prob'].sum()
    n = data.groupby('bin').size()
    
    hl_stat = ((observed - expected) ** 2 / (expected * (1 - expected / n))).sum()
    p_value = chi2.sf(hl_stat, n_bins - 2)
    
    return hl_stat, p_value

def calibration_plot(labels, predicted_probs, save_path, save_plot = False, n_bins=8):
    sns.set(style="whitegrid")
    plt.figure(figsize=(6, 6))
    
    # Compute calibration curve
    prob_true, prob_pred = calibration_curve(labels, predicted_probs, n_bins=n_bins)
    
    # Calculate standard deviation of predicted probabilities for each bin
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bins = np.digitize(predicted_probs, bin_edges) - 1
    
    # Ensure bins go from 0 to n_bins-1
    valid_bins = set(bins)
    std_devs = [np.std([predicted_probs[j] for j in range(len(predicted_probs)) if bins[j] == i]) for i in range(n_bins) if i in valid_bins]
    
    # Plot calibration points with error bars
    plt.errorbar(prob_pred, prob_true, yerr=std_devs, fmt='o', label='Observations', color='royalblue', ecolor='lightblue', capsize=5)
    
    # Fit a linear regression model
    lr = LinearRegression().fit(prob_pred.reshape(-1, 1), prob_true)
    intercept = lr.intercept_
    slope = lr.coef_[0]
    r_squared = lr.score(prob_pred.reshape(-1, 1), prob_true)
    
    # Generate points for the fitted line
    x_range = np.linspace(0, 1, 100)
    fitted_line = intercept + slope * x_range
    
    # Plot the perfectly calibrated line
    plt.plot([0, 1], [0, 1], linestyle='--', label='Perfectly Calibrated', color='gray', linewidth=2)
    
    # Plot the linear regression line
    plt.plot(x_range, fitted_line, label='Fitted Line', color='firebrick', linewidth=2)
    
    # Add intercept, slope, and R^2 to the plot
    plt.text(0.95, 0.15, f'$R^2$: {r_squared:.2f}', fontsize=16, verticalalignment='bottom', horizontalalignment='right', transform=plt.gca().transAxes, color='black')
    plt.text(0.95, 0.10, f'Intercept: {intercept:.2f}', fontsize=16, verticalalignment='bottom', horizontalalignment='right', transform=plt.gca().transAxes, color='black')
    plt.text(0.95, 0.05, f'Slope: {slope:.2f}', fontsize=16, verticalalignment='bottom', horizontalalignment='right', transform=plt.gca().transAxes, color='black')
    
    """
    # Add Brier score
    brier_score = brier_score_loss(labels, predicted_probs)
    plt.text(0.95, 0.05, f'Brier Score: {brier_score:.3f}', fontsize=12, verticalalignment='bottom', horizontalalignment='right', transform=plt.gca().transAxes, color='black')

    
    # Add KS test p-value with conditional formatting
    df = pd.DataFrame({"label": labels, "prob": predicted_probs})
    class0 = df[df['label'] == 0]
    class1 = df[df['label'] == 1]
    ks = ks_2samp(class0['prob'], class1['prob'])
    
    hl_stat, hl_p_value = hosmer_lemeshow_test(labels, predicted_probs, n_bins)
    plt.text(0.95, 0.05, f'HL: {hl_stat:.2f} (p-value: {format_p_value(hl_p_value)})', fontsize=12, verticalalignment='bottom', horizontalalignment='right', transform=plt.gca().transAxes, color='black')
    
    """
    # Add labels and title
    plt.xlabel('Predicted Probability', fontsize=14)
    plt.ylabel('True Probability', fontsize=14)
    plt.title('2-Year DFS Calibration Plot', fontsize=16)
    
    # Add grid lines
    plt.grid(True, linestyle='--', alpha=0.7)
    
    # Customize the legend
    plt.legend(fontsize=12, loc='upper left')
    
    # Set axis limits
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    
    # Show plot
    plt.tight_layout()
    if save_plot:
        plt.savefig(save_path, dpi=400)
    plt.show()
    plt.close()

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, precision_score, recall_score

def calculate_metrics(labels, probs, average='macro'):
    
    outputs = [0 if i < 0.5 else 1 for i in probs]
    
    f1 = f1_score(labels, outputs, average=average)
    accuracy = accuracy_score(labels, outputs)
    auc = roc_auc_score(labels, probs, multi_class='ovr')  # Adjust if multiclass
    precision = precision_score(labels, outputs, average=average)
    recall = recall_score(labels, outputs, average=average)
    
    return {
        'F1 Score Macro': f1,
        'Accuracy': accuracy,
        'AUC OVR': auc,
        'Precision Macro': precision,
        'Recall Macro': recall,
    }
    